# MARID ESKF Cascade Correction Evaluation

Tests **three configurations** on the velocity-LSTM held-out val flights (cold-start, no data leakage):

| Config | Description |
|--------|-------------|
| **A** | Raw ESKF velocity — dead-reckoning from GT pos at t=0 |
| **B** | + Yaw FF — Δψ rotates world-frame velocity before integration |
| **C** | + Vel LSTM — LSTM Δvx/Δvy applied to **raw** ESKF velocity |

> **Why no cascade (Yaw FF + LSTM)?**  
> The velocity LSTM was trained on raw ESKF errors — which are dominated by uncorrected yaw error.  
> Applying the LSTM on top of yaw-corrected velocity feeds it contradicting pose, causing double-counting and degraded performance. Removed.

**Metrics per config per flight:**
- Yaw RMSE (deg)
- Velocity RMSE: vx, vy, combined (m/s)
- Dead-reckoning position error at 30 s, 60 s, 120 s, 300 s, end-of-flight (m)
- Position drift rate (m/min, linear fit on Euclidean error vs time)

In [ ]:
import json
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from pathlib import Path
%matplotlib inline

# ── Configuration ─────────────────────────────────────────────────────────
DATA_DIR  = Path('~/marid_ws/data_extended').expanduser()
OUT_JSON  = DATA_DIR / 'cascade_eval_results.json'

DT        = 1.0 / 50.0   # 50 Hz logging rate [s]
CHUNK_LEN = 300           # LSTM inference chunk (must match training CHUNK_LEN)

# Val flights held out from velocity LSTM training (cold-start, no data leakage)
VAL_FLIGHTS = [
    'flight_20260521_211150',   #  8.9 min, 2D spread 1041×1304 m, yaw err ~4.9°
    'flight_20260522_062211',   # 14.6 min, 2D spread 2546×2365 m, yaw err ~4.8°
    'flight_20260523_090404',   # 29.1 min, 2D spread 4136×3180 m, yaw err ~3.7°
]

# Snapshot times for position error table [s]
SNAPSHOT_TIMES = [30, 60, 120, 300]

print(f'DATA_DIR: {DATA_DIR}')
print(f'Val flights: {len(VAL_FLIGHTS)}')

## Feature Augmentation & Data Loading

In [ ]:
_G        = 9.81
_MIN_V    = 3.0
_PDOT_MAX = 2.0

def _wrap(a):
    return ((a + np.pi) % (2 * np.pi) - np.pi).astype(np.float32)


def _augment_eskf_inputs(eskf: np.ndarray, d: dict) -> np.ndarray:
    """23-col augmented input — identical to both training scripts."""
    N           = len(eskf)
    thrust      = d['thrust'].astype(np.float32).ravel()[:N] if 'thrust' in d else np.zeros(N, np.float32)
    z           = eskf[:, 2]
    roll        = eskf[:, 3]
    pitch       = eskf[:, 4]
    yaw_eskf    = eskf[:, 5]
    vx, vy      = eskf[:, 6], eskf[:, 7]
    ground_flag = (z < 1.0).astype(np.float32)

    if 'airspeed' in d:
        V_src = np.clip(np.abs(d['airspeed'].astype(np.float32).ravel()[:N]), _MIN_V, None)
    else:
        V_src = np.clip(np.sqrt(vx**2 + vy**2), _MIN_V, None)

    psi_dot = np.clip((_G / V_src) * np.tan(roll), -_PDOT_MAX, _PDOT_MAX).astype(np.float32)
    psi_dot *= (1.0 - ground_flag)

    base = np.concatenate([eskf, thrust[:, None], ground_flag[:, None], psi_dot[:, None]], axis=1)

    if 'imu_acc' not in d:
        return base   # 15 cols (legacy)

    imu_acc          = d['imu_acc'].astype(np.float32)[:N]
    imu_acc[:, 1]    = np.clip(imu_acc[:, 1], -15.0, 15.0)
    a_excess         = (np.linalg.norm(imu_acc, axis=1) - _G).astype(np.float32)
    imu_acc[:, 2]   -= (_G * np.cos(roll) * np.cos(pitch)).astype(np.float32)
    yaw_madgwick     = d['yaw_madgwick'].astype(np.float32).ravel()[:N]
    airspeed         = d['airspeed'].astype(np.float32).ravel()[:N]
    sun_yaw          = d['sun_yaw'].astype(np.float32).ravel()[:N]
    sun_valid        = d['sun_valid'].astype(np.float32).ravel()[:N]

    delta_madgwick   = _wrap(yaw_madgwick - yaw_eskf)
    delta_sun        = _wrap(sun_yaw      - yaw_eskf) * sun_valid

    return np.concatenate([base,
                           imu_acc,
                           a_excess[:, None],
                           delta_madgwick[:, None],
                           airspeed[:, None],
                           delta_sun[:, None],
                           sun_valid[:, None]], axis=1)   # 23 cols


def load_flight(fid: str):
    """Load and concatenate all chunks for a flight.

    Returns
    -------
    feat   : (T, 23) float32   augmented ESKF input
    eskf   : (T, 12) float32   raw ESKF state
    gt_pos : (T, 2)  float32   [x_gt, y_gt] m
    gt_yaw : (T,)    float32   yaw_gt rad
    gt_vel : (T, 2)  float32   [vx_gt, vy_gt] m/s  (world frame)
    """
    chunks = sorted(DATA_DIR.glob(f'marid_eskf_gt_{fid}_chunk*.npz'))
    if not chunks:
        raise FileNotFoundError(f'No chunks found for {fid} in {DATA_DIR}')

    feat_parts, eskf_parts, gt_pos_parts, gt_yaw_parts, gt_vel_parts = [], [], [], [], []
    for p in chunks:
        d    = np.load(p, allow_pickle=True)
        eskf = d['eskf_inputs'].astype(np.float32)
        gt   = d['pose_targets'].astype(np.float32)   # (N, 7): [x,y,roll,pitch,yaw,vx,vy]

        feat_parts.append(_augment_eskf_inputs(eskf, d))
        eskf_parts.append(eskf)
        gt_pos_parts.append(gt[:, 0:2])
        gt_yaw_parts.append(gt[:, 4])
        gt_vel_parts.append(gt[:, 5:7])

    return (np.concatenate(feat_parts),
            np.concatenate(eskf_parts),
            np.concatenate(gt_pos_parts),
            np.concatenate(gt_yaw_parts),
            np.concatenate(gt_vel_parts))

print('Helpers defined.')

## Model Definitions & Loading

In [ ]:
# ── Model definitions (must match training scripts exactly) ───────────────

def make_attitude_ff(input_dim: int = 23) -> nn.Sequential:
    """Matches make_model() in eskf_attitude_ff_train.py exactly (bare Sequential)."""
    return nn.Sequential(
        nn.Linear(input_dim, 256), nn.ReLU(), nn.Dropout(0.3),
        nn.Linear(256,       256), nn.ReLU(), nn.Dropout(0.3),
        nn.Linear(256,         1),
    )


class VelocityLSTM(nn.Module):
    def __init__(self, input_dim=23, hidden_dim=128, num_layers=2, output_dim=2):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers,
                            batch_first=True, dropout=0.3)
        self.drop = nn.Dropout(0.4)
        self.fc   = nn.Linear(hidden_dim, output_dim)

    def forward(self, x, hidden=None):
        out, hidden = self.lstm(x, hidden)
        return self.fc(self.drop(out)), hidden


# ── Load models ───────────────────────────────────────────────────────────

# Attitude FF
_ff_norm  = np.load(DATA_DIR / 'eskf_attitude_ff_norm.npz')
ff_X_mean = _ff_norm['X_mean'].astype(np.float32)
ff_X_std  = _ff_norm['X_std'].astype(np.float32)
ff_y_mean = float(_ff_norm['y_mean'])
ff_y_std  = float(_ff_norm['y_std'])

att_model = make_attitude_ff()
att_model.load_state_dict(torch.load(DATA_DIR / 'eskf_attitude_ff.pt', map_location='cpu', weights_only=True))
att_model.eval()
print(f'Attitude FF loaded  — y_mean={ff_y_mean:.4f} rad  y_std={ff_y_std:.4f} rad')

# Velocity LSTM
_vel_norm  = np.load(DATA_DIR / 'eskf_velocity_lstm_norm.npz')
vel_X_mean = _vel_norm['X_mean'].astype(np.float32)
vel_X_std  = _vel_norm['X_std'].astype(np.float32)
vel_y_mean = _vel_norm['y_mean'].astype(np.float32)   # (2,)
vel_y_std  = _vel_norm['y_std'].astype(np.float32)    # (2,)

vel_model = VelocityLSTM()
vel_model.load_state_dict(torch.load(DATA_DIR / 'eskf_velocity_lstm.pt', map_location='cpu', weights_only=True))
vel_model.eval()
print(f'Velocity LSTM loaded — y_mean={vel_y_mean}  y_std={vel_y_std}')

## Inference Helpers

In [ ]:
def predict_yaw_correction(feat: np.ndarray) -> np.ndarray:
    """Δψ for every timestep [rad], pointwise (no hidden state)."""
    X = torch.tensor((feat - ff_X_mean) / ff_X_std)
    with torch.no_grad():
        y_norm = att_model(X).numpy().ravel()
    return (y_norm * ff_y_std + ff_y_mean).astype(np.float32)


def predict_velocity_correction(feat: np.ndarray) -> np.ndarray:
    """[Δvx, Δvy] for every timestep [m/s]. LSTM cold-started (h=None).

    Processes the full flight in CHUNK_LEN=300 chunks, carrying hidden state
    forward — the real deployment scenario (no warm-up from GT data).
    """
    T      = len(feat)
    delta  = np.zeros((T, 2), dtype=np.float32)
    X_norm = ((feat - vel_X_mean) / vel_X_std).astype(np.float32)

    hidden = None
    with torch.no_grad():
        for i in range(0, T, CHUNK_LEN):
            chunk = torch.tensor(X_norm[i:i + CHUNK_LEN]).unsqueeze(0)  # (1, L, 23)
            pred, hidden = vel_model(chunk, hidden)
            # Clamp cell state to prevent float32 overflow on long flights
            hidden = (hidden[0].detach(),
                      torch.clamp(hidden[1].detach(), -100.0, 100.0))
            raw = pred.squeeze(0).numpy()   # (L, 2) normalised
            delta[i:i + len(raw)] = raw * vel_y_std + vel_y_mean

    return delta


def rotate_velocity(vx: np.ndarray, vy: np.ndarray,
                    dpsi: np.ndarray) -> tuple:
    """Rotate world-frame velocity by Δψ.

    If ESKF yaw has error Δψ = ψ_true − ψ_eskf, the corrected velocity is:
        v_corr = R(Δψ) @ v_eskf
    """
    c, s = np.cos(dpsi), np.sin(dpsi)
    return c * vx - s * vy, s * vx + c * vy


def dead_reckon(pos0: np.ndarray, vx: np.ndarray, vy: np.ndarray) -> np.ndarray:
    """Euler-integrate velocity from pos0 at 50 Hz → (T, 2) position array."""
    T   = len(vx)
    pos = np.empty((T, 2), dtype=np.float32)
    pos[0] = pos0
    for t in range(1, T):
        pos[t, 0] = pos[t-1, 0] + vx[t-1] * DT
        pos[t, 1] = pos[t-1, 1] + vy[t-1] * DT
    return pos

print('Inference helpers defined.')

## Evaluate Val Flights

For each flight: run all three configs, compute yaw RMSE, velocity RMSE, and dead-reckoning position error.

In [ ]:
def eval_flight(fid: str) -> dict:
    print(f'\n{"─"*60}')
    print(f'Flight: {fid}')
    print(f'{"─"*60}')

    feat, eskf, gt_pos, gt_yaw, gt_vel = load_flight(fid)
    T         = len(feat)
    duration  = T * DT
    print(f'  Steps: {T:,}  ({duration/60:.1f} min)')

    yaw_eskf = eskf[:, 5]
    vx_eskf  = eskf[:, 6]
    vy_eskf  = eskf[:, 7]
    eskf_pos = eskf[:, 0:2]   # actual sensor-fusion position (FAST-LIO/baro/sonar)

    # ── Model predictions ─────────────────────────────────────────────────
    dpsi  = predict_yaw_correction(feat)         # (T,) rad
    dvel  = predict_velocity_correction(feat)    # (T, 2) m/s

    yaw_corr = yaw_eskf + dpsi

    # Config B: Yaw FF only — rotate world-frame velocity by Δψ
    vx_B, vy_B = rotate_velocity(vx_eskf, vy_eskf, dpsi)

    # Config C: Velocity LSTM on RAW ESKF velocity.
    vx_C = vx_eskf + dvel[:, 0]
    vy_C = vy_eskf + dvel[:, 1]

    # ── Dead reckoning (GPS-denied scenario) ──────────────────────────────
    # IMPORTANT: gt_pos is published at ~10-17 Hz (3-9 identical rows at 50 Hz
    # logging rate). Comparing ∫eskf_vel against raw gt_pos inflates error ~4x.
    # Correct reference: integrate gt_vel from the same starting point (pos0).
    # Then both dead-reckoned trajectories share the same physics.
    pos0       = gt_pos[0]
    pos_gt_dr  = dead_reckon(pos0, gt_vel[:, 0], gt_vel[:, 1])   # GT dead-reckoning reference
    pos_A      = dead_reckon(pos0, vx_eskf, vy_eskf)
    pos_B      = dead_reckon(pos0, vx_B,    vy_B)
    pos_C      = dead_reckon(pos0, vx_C,    vy_C)

    # ── ESKF sensor-fusion position RMSE vs Gazebo gt_pos ─────────────────
    # gt_pos is subsampled (~10-17 Hz) but still a valid absolute reference.
    eskf_pos_rmse = float(np.sqrt(np.mean(np.sum((eskf_pos - gt_pos)**2, axis=1))))

    # ── Yaw RMSE ──────────────────────────────────────────────────────────
    yaw_rmse_A = float(np.sqrt(np.mean(_wrap(yaw_eskf - gt_yaw)**2)))
    yaw_rmse_B = float(np.sqrt(np.mean(_wrap(yaw_corr  - gt_yaw)**2)))

    # ── Velocity RMSE ─────────────────────────────────────────────────────
    def vel_rmse(vx_est, vy_est):
        evx   = float(np.sqrt(np.mean((gt_vel[:, 0] - vx_est)**2)))
        evy   = float(np.sqrt(np.mean((gt_vel[:, 1] - vy_est)**2)))
        ecomb = float(np.sqrt(np.mean((gt_vel[:, 0] - vx_est)**2 +
                                      (gt_vel[:, 1] - vy_est)**2)))
        return evx, evy, ecomb

    vrmse_A = vel_rmse(vx_eskf, vy_eskf)
    vrmse_B = vel_rmse(vx_B,    vy_B)
    vrmse_C = vel_rmse(vx_C,    vy_C)

    # ── Position error vs GT dead-reckoning reference ─────────────────────
    def pos_err_at(pos_est, t_sec):
        idx = min(int(t_sec / DT), T - 1)
        return float(np.linalg.norm(pos_est[idx] - pos_gt_dr[idx]))

    def pos_rmse_full(pos_est):
        return float(np.sqrt(np.mean(np.sum((pos_est - pos_gt_dr)**2, axis=1))))

    def drift_rate(pos_est):
        """Linear fit on Euclidean error vs time [m/min]."""
        err  = np.linalg.norm(pos_est - pos_gt_dr, axis=1)
        t_m  = np.arange(T) * DT / 60.0
        slope, _ = np.polyfit(t_m, err, 1)
        return float(max(slope, 0.0))

    # Theoretical max drift = vel_rmse * T (bounding lemma: ∫|Δv|dt ≥ |∫Δv dt|)
    vel_rmse_A_val = vrmse_A[2]
    theoretical_max = vel_rmse_A_val * duration

    snapshots = {}
    for cfg_name, pos_est in [('A', pos_A), ('B', pos_B), ('C', pos_C)]:
        s = {f'{t}s': pos_err_at(pos_est, t) for t in SNAPSHOT_TIMES}
        s['end']                   = pos_err_at(pos_est, duration)
        s['rmse_full']             = pos_rmse_full(pos_est)
        s['drift_rate_m_per_min']  = drift_rate(pos_est)
        snapshots[cfg_name] = s

    # ── Print per-flight summary ──────────────────────────────────────────
    print(f'\n  ESKF sensor-fusion position RMSE: {eskf_pos_rmse:.2f} m  (vs Gazebo gt_pos)')
    print(f'  Velocity RMSE bound on dead-reckoning: {theoretical_max:.1f} m  '
          f'(= {vel_rmse_A_val:.3f} m/s × {duration:.0f} s)')

    print(f'\n  Yaw RMSE:')
    print(f'    A  ESKF only : {np.degrees(yaw_rmse_A):6.2f}°')
    print(f'    B  + Yaw FF  : {np.degrees(yaw_rmse_B):6.2f}°  '
          f'({100*(yaw_rmse_A-yaw_rmse_B)/yaw_rmse_A:+.1f}%)')

    print(f'\n  Velocity RMSE (vx, vy, combined):')
    print(f'    A  ESKF only      : {vrmse_A[0]:.3f}  {vrmse_A[1]:.3f}  {vrmse_A[2]:.3f} m/s')
    print(f'    B  + Yaw FF       : {vrmse_B[0]:.3f}  {vrmse_B[1]:.3f}  {vrmse_B[2]:.3f} m/s  '
          f'({100*(vrmse_A[2]-vrmse_B[2])/vrmse_A[2]:+.1f}%)')
    print(f'    C  + Vel LSTM(raw): {vrmse_C[0]:.3f}  {vrmse_C[1]:.3f}  {vrmse_C[2]:.3f} m/s  '
          f'({100*(vrmse_A[2]-vrmse_C[2])/vrmse_A[2]:+.1f}%)')

    print(f'\n  Dead-reckoning error vs ∫gt_vel (velocity-bounded reference):')
    hdr = f'    {"Config":<18}' + ''.join(f'{t}s'.rjust(8) for t in SNAPSHOT_TIMES) \
          + '  End(m)  RMSE(m)  Drift(m/min)'
    print(hdr)
    for cfg, label in [('A', 'A  ESKF only'), ('B', 'B  + Yaw FF'), ('C', 'C  + Vel LSTM(raw)')]:
        s    = snapshots[cfg]
        cols = ''.join(f'{s[f"{t}s"]:7.1f} ' for t in SNAPSHOT_TIMES)
        print(f'    {label:<18}{cols}  {s["end"]:6.1f}   {s["rmse_full"]:6.1f}   '
              f'{s["drift_rate_m_per_min"]:.2f}')

    return {
        'duration_min':      duration / 60.0,
        'eskf_pos_rmse_m':   eskf_pos_rmse,
        'yaw_rmse_deg':      {'A': np.degrees(yaw_rmse_A), 'B': np.degrees(yaw_rmse_B)},
        'vel_rmse_mps': {
            'A': {'vx': vrmse_A[0], 'vy': vrmse_A[1], 'combined': vrmse_A[2]},
            'B': {'vx': vrmse_B[0], 'vy': vrmse_B[1], 'combined': vrmse_B[2]},
            'C': {'vx': vrmse_C[0], 'vy': vrmse_C[1], 'combined': vrmse_C[2]},
        },
        'pos_snapshots': snapshots,
        '_arrays': {
            'T': T, 'gt_pos': gt_pos, 'gt_vel': gt_vel,
            'eskf_pos':  eskf_pos,             # sensor-fusion position (vs gt_pos)
            'pos_gt_dr': pos_gt_dr,            # ∫gt_vel — correct DR reference
            'pos_A': pos_A, 'pos_B': pos_B, 'pos_C': pos_C,
            'vx_A': vx_eskf, 'vy_A': vy_eskf,
            'vx_B': vx_B,    'vy_B': vy_B,
            'vx_C': vx_C,    'vy_C': vy_C,
            'yaw_eskf': yaw_eskf, 'yaw_corr': yaw_corr, 'gt_yaw': gt_yaw,
        },
    }


# ── Run all val flights ────────────────────────────────────────────────────
results = {}
for fid in VAL_FLIGHTS:
    try:
        results[fid] = eval_flight(fid)
    except FileNotFoundError as e:
        print(f'\nWARNING: {e} — skipping')

if not results:
    raise RuntimeError('No results — check DATA_DIR and VAL_FLIGHTS list.')


## Aggregate Summary — All Val Flights

In [ ]:
configs = ['A', 'B', 'C']
labels  = {
    'A': 'A  ESKF only       ',
    'B': 'B  + Yaw FF        ',
    'C': 'C  + Vel LSTM(raw) ',
}

print(f'{"═"*65}')
print('AGGREGATE SUMMARY — all val flights (mean ± std)')
print(f'{"═"*65}')

print('\nYaw RMSE (deg):')
for cfg in ['A', 'B']:
    vals = [r['yaw_rmse_deg'][cfg] for r in results.values()]
    print(f'  {labels[cfg]}: {np.mean(vals):.2f}° ± {np.std(vals):.2f}°')

print('\nVelocity RMSE combined (m/s):')
for cfg in configs:
    vals = [r['vel_rmse_mps'][cfg]['combined'] for r in results.values()]
    imp  = [(r['vel_rmse_mps']['A']['combined'] - r['vel_rmse_mps'][cfg]['combined'])
            / r['vel_rmse_mps']['A']['combined'] * 100 for r in results.values()]
    print(f'  {labels[cfg]}: {np.mean(vals):.3f} ± {np.std(vals):.3f} m/s  '
          f'({np.mean(imp):+.1f}% vs A)')

print('\nPosition drift rate (m/min):')
for cfg in configs:
    vals = [r['pos_snapshots'][cfg]['drift_rate_m_per_min'] for r in results.values()]
    imp  = [(r['pos_snapshots']['A']['drift_rate_m_per_min'] -
             r['pos_snapshots'][cfg]['drift_rate_m_per_min'])
            / max(r['pos_snapshots']['A']['drift_rate_m_per_min'], 1e-3) * 100
            for r in results.values()]
    print(f'  {labels[cfg]}: {np.mean(vals):.2f} ± {np.std(vals):.2f} m/min  '
          f'({np.mean(imp):+.1f}% vs A)')

# ── Save JSON ─────────────────────────────────────────────────────────────
def _serialise(r):
    return {k: v for k, v in r.items() if k != '_arrays'}

OUT_JSON.write_text(json.dumps({fid: _serialise(r) for fid, r in results.items()}, indent=2))
print(f'\nResults saved → {OUT_JSON}')

## Overview Plots

Three columns per flight:
1. **XY trajectory** — ground truth vs **actual ESKF sensor-fusion position** (maintained by FAST-LIO/baro/sonar, ~20 m RMSE)
2. **Velocity error vs time** — combined |Δv| for A / B / C with 10 s rolling mean
3. **GPS-denied dead-reckoning error** — what happens if you integrate the (corrected) velocity from the last known GT position; diverges because ESKF vx/vy is airspeed-derived, not groundspeed

> ⚠️ Panel 1 shows **sensor-fusion position** (accurate).  
> ⚠️ Panel 3 shows **dead-reckoning** (diverges — this is the GPS-denied scenario, not ESKF failure).

In [ ]:
colors = {'A': '#888888', 'B': '#2196F3', 'C': '#E91E63', 'GT_DR': '#4CAF50'}

n_flights = len(results)
fig, axes = plt.subplots(n_flights, 3, figsize=(18, 6 * n_flights))
if n_flights == 1:
    axes = axes[np.newaxis, :]

W = int(10.0 / DT)    # 10-second rolling window

for row, (fid, res) in enumerate(results.items()):
    arr       = res['_arrays']
    T         = arr['T']
    t_min     = np.arange(T) * DT / 60.0
    gt_p      = arr['gt_pos']
    gt_v      = arr['gt_vel']
    eskf_p    = arr['eskf_pos']
    pos_gt_dr = arr['pos_gt_dr']

    # ── Col 0: XY — GT vs ESKF sensor-fusion (actual flight footprint) ────
    # Dead-reckoned paths (pos_A/B/C) are NOT shown here.
    # Although velocity is world-frame, the drone's circuit/back-and-forth pattern
    # means total path length ≈ 3.9× net displacement.  Integrating velocity
    # traces ~70 km of path over 29 min, while the GT footprint is ~4×3 km —
    # incompatible scales that make XY overlay meaningless.
    # GPS-denied navigation quality is shown correctly in Col 2 (error vs time).
    ax = axes[row, 0]
    ax.plot(gt_p[:, 0],   gt_p[:, 1],   'k-',  lw=1.8, label='GT  (Gazebo pos)',   alpha=0.85)
    ax.plot(eskf_p[:, 0], eskf_p[:, 1], '--',  lw=1.1, color=colors['A'],
            label=f'ESKF fusion  RMSE = {res["eskf_pos_rmse_m"]:.1f} m', alpha=0.9)
    ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)')
    ax.set_title(f'{fid}\nESKF sensor-fusion position vs GT')
    ax.legend(fontsize=8); ax.set_aspect('equal'); ax.grid(True, alpha=0.3)

    # ── Col 1: Velocity error A / B / C with 10 s rolling mean ───────────
    ax = axes[row, 1]
    for vx_k, vy_k, cfg in [('vx_A','vy_A','A'), ('vx_B','vy_B','B'), ('vx_C','vy_C','C')]:
        ve      = np.sqrt((arr[vx_k] - gt_v[:, 0])**2 + (arr[vy_k] - gt_v[:, 1])**2)
        ve_roll = np.convolve(ve, np.ones(W) / W, mode='valid')
        t_roll  = t_min[W-1:]
        rmse    = res['vel_rmse_mps'][cfg]['combined']
        ax.plot(t_roll, ve_roll, color=colors[cfg], lw=1.1,
                label=f'{cfg}  RMSE = {rmse:.3f} m/s')
    ax.set_xlabel('Time (min)'); ax.set_ylabel('|Δv| combined (m/s)')
    ax.set_title('Velocity error — 10 s rolling mean\nA vs B (+Yaw FF) vs C (+Vel LSTM)')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    # ── Col 2: GPS-denied dead-reckoning error vs ∫gt_vel ─────────────────
    # pos_A/B/C all start at gt_pos[0] and integrate their respective velocities.
    # Reference = pos_gt_dr = ∫gt_vel (same integration, ideal velocity).
    # Error is therefore bounded by velocity RMSE × T (consistent with Col 1).
    ax = axes[row, 2]
    for cfg, pk in [('A', 'pos_A'), ('B', 'pos_B'), ('C', 'pos_C')]:
        err = np.linalg.norm(arr[pk] - pos_gt_dr, axis=1)
        dr  = res['pos_snapshots'][cfg]['drift_rate_m_per_min']
        ax.plot(t_min, err, color=colors[cfg], lw=0.9,
                label=f'{cfg}  {dr:.0f} m/min drift')
    ax.set_xlabel('Time (min)'); ax.set_ylabel('Dead-reckoning error (m)')
    ax.set_title('GPS-denied DR error vs ∫gt_vel\n(velocity-bounded — path length ≈ 3.9× displacement)')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.suptitle('MARID ESKF Cascade Correction Evaluation\n'
             'A = ESKF baseline   B = +Yaw FF   C = +Vel LSTM (raw ESKF inputs)',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


## Yaw Correction Detail

Per-flight: ESKF yaw vs corrected yaw vs ground-truth yaw, and the Δψ signal fed into the velocity rotation.

In [ ]:
for fid, res in results.items():
    arr   = res['_arrays']
    T     = arr['T']
    t_s   = np.arange(T) * DT
    t_min = t_s / 60.0

    yaw_eskf = arr['yaw_eskf']
    yaw_corr = arr['yaw_corr']
    gt_yaw   = arr['gt_yaw']
    dpsi     = _wrap(yaw_corr - yaw_eskf)   # Δψ applied by yaw FF

    fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

    axes[0].plot(t_min, np.degrees(gt_yaw),   'k-',  lw=1.4, label='GT yaw',       alpha=0.8)
    axes[0].plot(t_min, np.degrees(yaw_eskf), '-',   lw=0.9, color='#888888',      label='ESKF yaw (A)')
    axes[0].plot(t_min, np.degrees(yaw_corr), '--',  lw=1.0, color=colors['B'],    label='Corrected yaw (B)')
    axes[0].set_ylabel('Yaw (deg)'); axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)
    axes[0].set_title(f'{fid} — Yaw comparison')

    yaw_err_A = np.degrees(_wrap(yaw_eskf - gt_yaw))
    yaw_err_B = np.degrees(_wrap(yaw_corr  - gt_yaw))
    axes[1].plot(t_min, yaw_err_A, color='#888888', lw=0.8, label=f'ESKF yaw error  RMSE={res["yaw_rmse_deg"]["A"]:.2f}°')
    axes[1].plot(t_min, yaw_err_B, color=colors['B'], lw=0.9, label=f'Corrected error RMSE={res["yaw_rmse_deg"]["B"]:.2f}°')
    axes[1].axhline(0, color='white', lw=0.8, ls=':')
    axes[1].set_ylabel('Yaw error (deg)'); axes[1].set_xlabel('Time (min)')
    axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

## Velocity Correction Detail

Per-flight vx and vy: ground truth, ESKF (A), Yaw-corrected (B), LSTM-corrected (C).

In [ ]:
for fid, res in results.items():
    arr   = res['_arrays']
    T     = arr['T']
    t_min = np.arange(T) * DT / 60.0
    gt_v  = arr['gt_vel']

    fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

    for ax, ch, key in [
        (axes[0], 'vx', 0),
        (axes[1], 'vy', 1),
    ]:
        vgt = gt_v[:, key]
        ax.plot(t_min, vgt,           'k-',  lw=1.2, label='GT',        alpha=0.7)
        ax.plot(t_min, arr[f'{ch}_A'], '-',  lw=0.8, color=colors['A'], label='A  ESKF')
        ax.plot(t_min, arr[f'{ch}_B'], '-',  lw=0.8, color=colors['B'], label='B  +YawFF')
        ax.plot(t_min, arr[f'{ch}_C'], '--', lw=1.0, color=colors['C'], label='C  +VelLSTM')
        ax.set_ylabel(f'{ch} (m/s)')
        rmse_a = res['vel_rmse_mps']['A'][ch]
        rmse_b = res['vel_rmse_mps']['B'][ch]
        rmse_c = res['vel_rmse_mps']['C'][ch]
        ax.set_title(f'{ch}:  A={rmse_a:.3f} m/s  B={rmse_b:.3f} m/s  C={rmse_c:.3f} m/s  (RMSE)')
        ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

    axes[-1].set_xlabel('Time (min)')
    fig.suptitle(f'{fid} — Velocity channels', fontweight='bold')
    plt.tight_layout()
    plt.show()


## Bar Chart — Per-Flight Velocity RMSE

In [ ]:
fids_short = [f[-6:] for f in results.keys()]
x = np.arange(len(results))
w = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
for i, (cfg, label, color) in enumerate([('A','A  ESKF','#888888'), ('B','B  +YawFF','#2196F3'), ('C','C  +VelLSTM','#E91E63')]):
    vals = [r['vel_rmse_mps'][cfg]['combined'] for r in results.values()]
    bars = ax.bar(x + (i - 1) * w, vals, w, label=label, color=color, alpha=0.85)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.02, f'{v:.2f}',
                ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(fids_short)
ax.set_ylabel('Combined velocity RMSE (m/s)')
ax.set_title('Velocity RMSE per val flight — A / B / C configs')
ax.legend(); ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# ── Aggregate numbers ─────────────────────────────────────────────────────
print('\nAggregate combined velocity RMSE (m/s):')
for cfg in ['A','B','C']:
    vals = [r['vel_rmse_mps'][cfg]['combined'] for r in results.values()]
    imp  = [(r['vel_rmse_mps']['A']['combined'] - r['vel_rmse_mps'][cfg]['combined'])
            / r['vel_rmse_mps']['A']['combined'] * 100 for r in results.values()]
    print(f'  {cfg}: {np.mean(vals):.3f} ± {np.std(vals):.3f} m/s  ({np.mean(imp):+.1f}% vs A)')